In [3]:
# ============================================================
# Finale Klassifikation Level II — Individual / CoEco / Landscape
# Modell: MobileCLIP-S1 + Linear Probe (Macro-F1 = 0.819)
# Input: non-human Bilder aus Level I
# ============================================================
import os
import shutil
import torch
import joblib
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import mobileclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

# --- MobileCLIP-S1 laden ---
weights_path = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"
model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=weights_path,
    device=device
)
model.eval()
print("Modell geladen: MobileCLIP-S1")

# --- Linear Probe laden ---
clf_path = r"C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison_IndividualCoEcoLandscape\linear_probe_MobileCLIP-S1_individual_coeco_landscape.joblib"
clf = joblib.load(clf_path)
print("Linear Probe geladen")


# ============================================================
# Konfiguration
# ============================================================
categories       = ["individual", "community_ecosystem", "landscape"]
BATCH_SIZE       = 32
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

full_dataset_folder = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE_vitl14_sorted\non_human"
output_csv          = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE.csv"
output_base         = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE_sorted"


# ============================================================
# Hilfsfunktionen
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError):
        return None


# ============================================================
# Alle Bilder sammeln
# ============================================================
full_image_paths = []
for root, _, files in os.walk(full_dataset_folder):
    for fname in sorted(files):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTENSIONS:
            full_image_paths.append(os.path.join(root, fname))

print(f"Bilder gefunden: {len(full_image_paths)}")


# ============================================================
# Embeddings extrahieren + Linear Probe anwenden
# ============================================================
results, confidences, valid_paths = [], [], []

for path in tqdm(full_image_paths, desc="Klassifiziere"):
    img = safe_open(path)
    if img is None:
        continue

    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    embedding = feat.cpu().numpy()

    pred = clf.predict(embedding)[0]
    conf = clf.predict_proba(embedding).max()

    results.append(pred)
    confidences.append(float(conf))
    valid_paths.append(path)


# ============================================================
# Ergebnisse speichern
# ============================================================
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

df_results = pd.DataFrame({
    "file_name":  [os.path.basename(p) for p in valid_paths],
    "file_path":  valid_paths,
    "pred_label": results,
    "confidence": confidences,
})
df_results.to_csv(output_csv, index=False)

print(f"\n✅ Klassifikation abgeschlossen.")
print(f"Gespeichert: {output_csv}")
print("\nVerteilung im Gesamtdatensatz:")
print(df_results["pred_label"].value_counts())
print(f"\nGesamt klassifiziert: {len(df_results)} Bilder")


# ============================================================
# Bilder nach Vorhersage in Ordner sortieren
# ============================================================
sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in tqdm(zip(valid_paths, results), total=len(valid_paths), desc="Kopiere Bilder"):
    dest = os.path.join(sorted_dirs[pred], os.path.basename(path))
    if not os.path.exists(dest):
        shutil.copy(path, dest)

print("\nBilder sortiert:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)} → {sorted_dirs[cat]}")

Gerät: cuda
Modell geladen: MobileCLIP-S1
Linear Probe geladen
Bilder gefunden: 5240


Klassifiziere: 100%|███████████████████████████████████████████████████████████████| 5240/5240 [08:20<00:00, 10.48it/s]



✅ Klassifikation abgeschlossen.
Gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE.csv

Verteilung im Gesamtdatensatz:
pred_label
landscape              3422
community_ecosystem    1301
individual              517
Name: count, dtype: int64

Gesamt klassifiziert: 5240 Bilder


Kopiere Bilder: 100%|█████████████████████████████████████████████████████████████| 5240/5240 [00:10<00:00, 515.14it/s]


Bilder sortiert:
  individual: 517 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE_sorted\individual
  community_ecosystem: 1301 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE_sorted\community_ecosystem
  landscape: 3422 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE_sorted\landscape
